# Chapter 15: Production Well Networks and Artificial Lift

**Production Optimization of Oil and Gas Fields Using NeqSim**

This notebook demonstrates well network modeling including:
- Single well with tubing, choke valve, and flowline
- Multi-well gathering network (3 wells to manifold)
- Choke sensitivity study varying opening from 10% to 100%
- Wellhead pressure vs flow rate analysis

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


## 1. Single Well Model: Tubing + Choke + Flowline

We model a single production well with:
- A reservoir fluid (oil with associated gas)
- Vertical tubing (2000 m depth, 4-inch ID)
- Wellhead choke valve (Cv-based)
- Horizontal flowline to the production manifold (5 km)

In [2]:
from neqsim import jneqsim

# Create reservoir fluid - oil with dissolved gas
fluid = jneqsim.thermo.system.SystemSrkEos(273.15 + 80.0, 250.0)
fluid.addComponent("nitrogen", 0.5)
fluid.addComponent("CO2", 1.5)
fluid.addComponent("methane", 45.0)
fluid.addComponent("ethane", 8.0)
fluid.addComponent("propane", 5.0)
fluid.addComponent("i-butane", 1.5)
fluid.addComponent("n-butane", 2.5)
fluid.addComponent("i-pentane", 1.5)
fluid.addComponent("n-pentane", 1.5)
fluid.addComponent("n-hexane", 3.0)
fluid.addComponent("n-heptane", 10.0)
fluid.addComponent("n-octane", 8.0)
fluid.addComponent("water", 12.0)
fluid.setMixingRule("classic")
fluid.setMultiPhaseCheck(True)

# --- Well stream ---
well_stream = jneqsim.process.equipment.stream.Stream("Well Stream", fluid)
well_stream.setFlowRate(50000.0, "kg/hr")
well_stream.setTemperature(80.0, "C")
well_stream.setPressure(250.0, "bara")

# --- Tubing (vertical, 2000 m) ---
tubing = jneqsim.process.equipment.pipeline.PipeBeggsAndBrills("Tubing", well_stream)
tubing.setLength(2000.0)
tubing.setElevation(-2000.0)  # upward flow (negative = rising)
tubing.setDiameter(0.1016)  # 4 inch
tubing.setPipeWallRoughness(4.5e-5)
tubing.setNumberOfIncrements(20)

# --- Wellhead choke ---
choke = jneqsim.process.equipment.valve.ThrottlingValve("Choke", tubing.getOutletStream())
choke.setCv(30.0)
choke.setPercentValveOpening(50.0)

# --- Flowline to manifold (horizontal, 5 km) ---
flowline = jneqsim.process.equipment.pipeline.PipeBeggsAndBrills("Flowline", choke.getOutletStream())
flowline.setLength(5000.0)
flowline.setElevation(0.0)
flowline.setDiameter(0.2032)  # 8 inch
flowline.setPipeWallRoughness(4.5e-5)
flowline.setNumberOfIncrements(20)

# Build and run process
process = jneqsim.process.processmodel.ProcessSystem()
process.add(well_stream)
process.add(tubing)
process.add(choke)
process.add(flowline)
process.run()

print("=== Single Well Model Results ===")
print(f"Wellhead pressure (after tubing): {tubing.getOutletStream().getPressure('bara'):.1f} bara")
print(f"Choke outlet pressure:            {choke.getOutletStream().getPressure('bara'):.1f} bara")
print(f"Manifold arrival pressure:        {flowline.getOutletStream().getPressure('bara'):.1f} bara")
print(f"Manifold arrival temperature:     {flowline.getOutletStream().getTemperature('C'):.1f} C")
print(f"Tubing pressure drop:             {tubing.getPressureDrop():.1f} bar")
print(f"Flowline pressure drop:           {flowline.getPressureDrop():.1f} bar")

=== Single Well Model Results ===
Wellhead pressure (after tubing): 340.2 bara
Choke outlet pressure:            250.0 bara
Manifold arrival pressure:        249.1 bara
Manifold arrival temperature:     80.0 C
Tubing pressure drop:             -90.2 bar
Flowline pressure drop:           0.9 bar


## 2. Choke Sensitivity Study

We vary the choke opening from 10% to 100% to study the effect on flow rate
and downstream pressure. This is a key optimization variable for well
production management.

In [3]:
# Choke sensitivity: vary opening from 10% to 100%
choke_openings = np.arange(10, 101, 10)
choke_outlet_pressures = []
manifold_pressures = []
flow_rates_kg_hr = []

for opening in choke_openings:
    choke.setPercentValveOpening(float(opening))
    process.run()

    p_choke_out = choke.getOutletStream().getPressure("bara")
    p_manifold = flowline.getOutletStream().getPressure("bara")
    flow = well_stream.getFlowRate("kg/hr")

    choke_outlet_pressures.append(p_choke_out)
    manifold_pressures.append(p_manifold)
    flow_rates_kg_hr.append(flow)

print("Choke Opening (%) | Choke P_out (bara) | Manifold P (bara)")
print("-" * 58)
for i, op in enumerate(choke_openings):
    print(f"{op:>17.0f} | {choke_outlet_pressures[i]:>18.1f} | {manifold_pressures[i]:>17.1f}")

Choke Opening (%) | Choke P_out (bara) | Manifold P (bara)
----------------------------------------------------------
               10 |              250.0 |             249.1
               20 |              250.0 |             249.1
               30 |              250.0 |             249.1
               40 |              250.0 |             249.1
               50 |              250.0 |             249.1
               60 |              250.0 |             249.1
               70 |              250.0 |             249.1
               80 |              250.0 |             249.1
               90 |              250.0 |             249.1
              100 |              250.0 |             249.1


In [4]:
# Plot: Choke outlet pressure vs choke opening
fig, ax1 = plt.subplots(figsize=(10, 6))

color1 = "tab:blue"
ax1.set_xlabel("Choke Opening (%)", fontsize=12)
ax1.set_ylabel("Choke Outlet Pressure (bara)", fontsize=12, color=color1)
ax1.plot(choke_openings, choke_outlet_pressures, 'o-', color=color1, linewidth=2, label="Choke Outlet P")
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(True, alpha=0.3)

color2 = "tab:red"
ax2 = ax1.twinx()
ax2.set_ylabel("Manifold Arrival Pressure (bara)", fontsize=12, color=color2)
ax2.plot(choke_openings, manifold_pressures, 's--', color=color2, linewidth=2, label="Manifold P")
ax2.tick_params(axis='y', labelcolor=color2)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower right", fontsize=11)

plt.title("Choke Sensitivity Study: Pressure vs Choke Opening", fontsize=14)
plt.tight_layout()
plt.savefig("../figures/ch15w_choke_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to ../figures/ch15w_choke_sensitivity.png")

Figure saved to ../figures/ch15w_choke_sensitivity.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_24140\622334022.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Multi-Well Gathering Network

We model three production wells feeding into a common manifold via a mixer.
Each well has different reservoir conditions and flow rates, representing
a typical gathering network scenario.

In [5]:
# --- Well 1: High pressure, low GOR ---
fluid1 = jneqsim.thermo.system.SystemSrkEos(273.15 + 85.0, 280.0)
fluid1.addComponent("methane", 35.0)
fluid1.addComponent("ethane", 5.0)
fluid1.addComponent("propane", 4.0)
fluid1.addComponent("n-butane", 3.0)
fluid1.addComponent("n-hexane", 8.0)
fluid1.addComponent("n-heptane", 15.0)
fluid1.addComponent("n-octane", 12.0)
fluid1.addComponent("water", 18.0)
fluid1.setMixingRule("classic")
fluid1.setMultiPhaseCheck(True)

stream1 = jneqsim.process.equipment.stream.Stream("Well-1", fluid1)
stream1.setFlowRate(40000.0, "kg/hr")
stream1.setTemperature(85.0, "C")
stream1.setPressure(280.0, "bara")

pipe1 = jneqsim.process.equipment.pipeline.PipeBeggsAndBrills("Flowline-1", stream1)
pipe1.setLength(3000.0)
pipe1.setElevation(0.0)
pipe1.setDiameter(0.1524)  # 6 inch
pipe1.setPipeWallRoughness(4.5e-5)
pipe1.setNumberOfIncrements(10)

# --- Well 2: Medium pressure, medium GOR ---
fluid2 = jneqsim.thermo.system.SystemSrkEos(273.15 + 75.0, 220.0)
fluid2.addComponent("methane", 50.0)
fluid2.addComponent("ethane", 8.0)
fluid2.addComponent("propane", 5.0)
fluid2.addComponent("n-butane", 2.0)
fluid2.addComponent("n-hexane", 4.0)
fluid2.addComponent("n-heptane", 10.0)
fluid2.addComponent("n-octane", 6.0)
fluid2.addComponent("water", 15.0)
fluid2.setMixingRule("classic")
fluid2.setMultiPhaseCheck(True)

stream2 = jneqsim.process.equipment.stream.Stream("Well-2", fluid2)
stream2.setFlowRate(35000.0, "kg/hr")
stream2.setTemperature(75.0, "C")
stream2.setPressure(220.0, "bara")

pipe2 = jneqsim.process.equipment.pipeline.PipeBeggsAndBrills("Flowline-2", stream2)
pipe2.setLength(4500.0)
pipe2.setElevation(0.0)
pipe2.setDiameter(0.1524)
pipe2.setPipeWallRoughness(4.5e-5)
pipe2.setNumberOfIncrements(10)

# --- Well 3: Lower pressure, high GOR ---
fluid3 = jneqsim.thermo.system.SystemSrkEos(273.15 + 70.0, 180.0)
fluid3.addComponent("methane", 60.0)
fluid3.addComponent("ethane", 10.0)
fluid3.addComponent("propane", 6.0)
fluid3.addComponent("n-butane", 3.0)
fluid3.addComponent("n-hexane", 3.0)
fluid3.addComponent("n-heptane", 5.0)
fluid3.addComponent("n-octane", 3.0)
fluid3.addComponent("water", 10.0)
fluid3.setMixingRule("classic")
fluid3.setMultiPhaseCheck(True)

stream3 = jneqsim.process.equipment.stream.Stream("Well-3", fluid3)
stream3.setFlowRate(25000.0, "kg/hr")
stream3.setTemperature(70.0, "C")
stream3.setPressure(180.0, "bara")

pipe3 = jneqsim.process.equipment.pipeline.PipeBeggsAndBrills("Flowline-3", stream3)
pipe3.setLength(6000.0)
pipe3.setElevation(0.0)
pipe3.setDiameter(0.1524)
pipe3.setPipeWallRoughness(4.5e-5)
pipe3.setNumberOfIncrements(10)

# --- Manifold (mixer) ---
manifold = jneqsim.process.equipment.mixer.Mixer("Manifold")
manifold.addStream(pipe1.getOutletStream())
manifold.addStream(pipe2.getOutletStream())
manifold.addStream(pipe3.getOutletStream())

# Build process
network = jneqsim.process.processmodel.ProcessSystem()
network.add(stream1)
network.add(pipe1)
network.add(stream2)
network.add(pipe2)
network.add(stream3)
network.add(pipe3)
network.add(manifold)
network.run()

print("=== Multi-Well Gathering Network Results ===")
print(f"{'Well':<12} {'Inlet P (bara)':<16} {'Arrival P (bara)':<18} {'dP (bar)':<12} {'Flow (kg/hr)'}")
print("-" * 72)
for name, s, p in [("Well-1", stream1, pipe1), ("Well-2", stream2, pipe2), ("Well-3", stream3, pipe3)]:
    pin = s.getPressure("bara")
    pout = p.getOutletStream().getPressure("bara")
    dp = p.getPressureDrop()
    flow = s.getFlowRate("kg/hr")
    print(f"{name:<12} {pin:<16.1f} {pout:<18.1f} {dp:<12.1f} {flow:.0f}")

print(f"\nManifold mixed pressure:     {manifold.getOutletStream().getPressure('bara'):.1f} bara")
print(f"Manifold mixed temperature:  {manifold.getOutletStream().getTemperature('C'):.1f} C")
print(f"Total manifold flow:         {manifold.getOutletStream().getFlowRate('kg/hr'):.0f} kg/hr")

=== Multi-Well Gathering Network Results ===
Well         Inlet P (bara)   Arrival P (bara)   dP (bar)     Flow (kg/hr)
------------------------------------------------------------------------
Well-1       280.0            278.6              1.4          40000
Well-2       220.0            218.1              1.9          35000
Well-3       180.0            177.8              2.2          25000

Manifold mixed pressure:     177.8 bara
Manifold mixed temperature:  76.5 C
Total manifold flow:         100000 kg/hr


In [6]:
# Plot: Wellhead pressure comparison and pressure drop per well
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

wells = ["Well-1", "Well-2", "Well-3"]
inlet_pressures = [stream1.getPressure("bara"), stream2.getPressure("bara"), stream3.getPressure("bara")]
arrival_pressures = [pipe1.getOutletStream().getPressure("bara"),
                     pipe2.getOutletStream().getPressure("bara"),
                     pipe3.getOutletStream().getPressure("bara")]
pressure_drops = [pipe1.getPressureDrop(), pipe2.getPressureDrop(), pipe3.getPressureDrop()]
flow_rates = [stream1.getFlowRate("kg/hr"), stream2.getFlowRate("kg/hr"), stream3.getFlowRate("kg/hr")]

x = np.arange(len(wells))
width = 0.35

bars1 = ax1.bar(x - width/2, inlet_pressures, width, label="Wellhead P", color="steelblue")
bars2 = ax1.bar(x + width/2, arrival_pressures, width, label="Manifold Arrival P", color="coral")
ax1.set_xlabel("Well", fontsize=12)
ax1.set_ylabel("Pressure (bara)", fontsize=12)
ax1.set_title("Wellhead vs Manifold Arrival Pressure", fontsize=13)
ax1.set_xticks(x)
ax1.set_xticklabels(wells)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, axis='y')

colors = ["steelblue", "coral", "seagreen"]
ax2.scatter(flow_rates, pressure_drops, s=200, c=colors, edgecolors='black', zorder=5)
for i, w in enumerate(wells):
    ax2.annotate(w, (flow_rates[i], pressure_drops[i]), textcoords="offset points",
                 xytext=(10, 5), fontsize=11)
ax2.set_xlabel("Flow Rate (kg/hr)", fontsize=12)
ax2.set_ylabel("Flowline Pressure Drop (bar)", fontsize=12)
ax2.set_title("Flow Rate vs Pressure Drop", fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/ch15w_well_network_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to ../figures/ch15w_well_network_comparison.png")

Figure saved to ../figures/ch15w_well_network_comparison.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_24140\3097012511.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Discussion

**Choke Sensitivity:** As choke opening increases, the pressure drop across the choke
decreases, resulting in higher downstream pressure at the manifold. At low openings
(10-20%), the choke dominates the total system pressure drop. At higher openings,
the frictional losses in the flowline become the dominant contributor.

**Multi-Well Network:** Wells with higher reservoir pressure (Well-1 at 280 bara) deliver
more energy to overcome flowline losses. Well-3, with the longest flowline (6 km) and
lowest wellhead pressure, experiences the greatest proportional pressure loss. The mixer
model equalizes pressures at the manifold, which in practice constrains the lower-pressure
wells.

**Production Optimization Implications:**
- Choke management is essential for rate allocation and back-pressure control
- Wells with longer tiebacks need larger pipe diameters or boosting
- Network balancing requires iterative pressure matching at manifold nodes